<a href="https://colab.research.google.com/github/Deepanshu-8126/Pandas/blob/main/Pandas_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pivot table + melt - data reshape


pivot table - rows to columns
and its opposite melt - convert wide table to long tablle

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv('/content/Sample - Superstore.csv', encoding = 'latin-1')

print(df.shape) # 9994 rows columns 21
print(df.columns.tolist())
print(df.head(3))

(9994, 21)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
   Row ID        Order ID Order Date   Ship Date     Ship Mode Customer ID  \
0       1  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
1       2  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
2       3  CA-2016-138688  6/12/2016   6/16/2016  Second Class    DV-13045   

     Customer Name    Segment        Country         City  ... Postal Code  \
0      Claire Gute   Consumer  United States    Henderson  ...       42420   
1      Claire Gute   Consumer  United States    Henderson  ...       42420   
2  Darrin Van Huff  Corporate  United States  Los Angeles  ...       90036   

   Region       Product ID         Category Sub-Category  \
0   South  FUR-BO-10001798        Furniture

In [ ]:
# 8.1 wide vs long format
# wide - long : melt  - columns to row
# long - wide - pivot  - row to column
# pivot_table -- pivot _ aggregation



In [9]:
# 8.2 pivot() - basic reshapping  - np aggregation

"""df.pivot(index='rows_column',
         columns='columns_column',
         values='values_column')"""

"df.pivot(index='rows_column',\n         columns='columns_column',\n         values='values_column')"

In [12]:
sample = pd.DataFrame({
    'Region':   ['West', 'West', 'East', 'East', 'South', 'South'],
    'Category': ['Furniture', 'Technology', 'Furniture', 'Technology',
                 'Furniture', 'Technology'],
    'Sales':    [1000, 2000, 1500, 2500, 800, 1200]
})
print(sample)

  Region    Category  Sales
0   West   Furniture   1000
1   West  Technology   2000
2   East   Furniture   1500
3   East  Technology   2500
4  South   Furniture    800
5  South  Technology   1200


In [13]:
# pivot - long to wide format
pivoted = sample.pivot(index = 'Region', columns = 'Category', values = 'Sales')
print(pivoted)

# unique values
# values - sales rows + column

""" Result:
        Row 'West' + Col 'Furniture'  → 1000
        Row 'West' + Col 'Technology' → 2000
        Row 'East' + Col 'Furniture'  → 1500  """

Category  Furniture  Technology
Region                         
East           1500        2500
South           800        1200
West           1000        2000


In [15]:
region_cat = df.groupby(['Region', 'Category'])['Sales'].sum().reset_index()
print(region_cat)


     Region         Category        Sales
0   Central        Furniture  163797.1638
1   Central  Office Supplies  167026.4150
2   Central       Technology  170416.3120
3      East        Furniture  208291.2040
4      East  Office Supplies  205516.0550
5      East       Technology  264973.9810
6     South        Furniture  117298.6840
7     South  Office Supplies  125651.3130
8     South       Technology  148771.9080
9      West        Furniture  252612.7435
10     West  Office Supplies  220853.2490
11     West       Technology  251991.8320


In [17]:
# then pivot
pivot_result = region_cat.pivot(index = 'Region', columns = 'Category', values = 'Sales').round(2)
print(pivot_result)

Category  Furniture  Office Supplies  Technology
Region                                          
Central   163797.16        167026.42   170416.31
East      208291.20        205516.06   264973.98
South     117298.68        125651.31   148771.91
West      252612.74        220853.25   251991.83


In [18]:
# pivot column names clean   ( label remove clean output )
pivot_result.columns.name = None
pivot_result.index.name = None

print(pivot_result)

         Furniture  Office Supplies  Technology
Central  163797.16        167026.42   170416.31
East     208291.20        205516.06   264973.98
South    117298.68        125651.31   148771.91
West     252612.74        220853.25   251991.83


In [21]:
# ⚠️ pivot() Error — Duplicate Values:
df_dup = pd.DataFrame({
    'Region':   ['West', 'West'],       # West repeat
    'Category': ['Furniture', 'Furniture'],  # Furniture repeat
    'Sales':    [1000, 500]
})
try:
    df_dup.pivot(index='Region', columns='Category', values='Sales')
except ValueError as e:
    print(f"Error: {e}")

Error: Index contains duplicate entries, cannot reshape


In [22]:
# solution for duplicates values
# 8.3 pivot_table - pivot + aggregation


# pivot_table() = pivot() + groupby()  combination
df.pivot_table(values='target_col',
               index='rows_col',
               columns='cols_col',
               aggfunc='sum',      # sum/mean/count/max/min
               fill_value=0)       # NaN ki jagah 0


KeyError: 'target_col'

In [24]:
pt = df.pivot_table(values = 'Sales', index = 'Region', columns = 'Category', aggfunc = 'sum').round(2)
print(pt)

Category  Furniture  Office Supplies  Technology
Region                                          
Central   163797.16        167026.42   170416.31
East      208291.20        205516.06   264973.98
South     117298.68        125651.31   148771.91
West      252612.74        220853.25   251991.83


In [28]:
# multi agg function
pt_multi = df.pivot_table(values = 'Sales', index = 'Region', columns = 'Category',
                          aggfunc = ['sum', 'mean']).round(2)
print(pt_multi)

                sum                                 mean                  \
Category  Furniture Office Supplies Technology Furniture Office Supplies   
Region                                                                     
Central   163797.16       167026.42  170416.31    340.53          117.46   
East      208291.20       205516.06  264973.98    346.57          120.04   
South     117298.68       125651.31  148771.91    353.31          126.28   
West      252612.74       220853.25  251991.83    357.30          116.42   

                     
Category Technology  
Region               
Central      405.75  
East         495.28  
South        507.75  
West         420.69  


In [30]:
# margins = True - row /column total
# rows and columns end total add
pt_margin = df.pivot_table(values='Sales',
                           index='Region',
                           columns='Category',
                           aggfunc='sum',
                           margins=True,
                           margins_name='TOTAL').round(2)
# clean  label
pt_margin.columns.name = None
pt_margin.index.name = None
print(pt_margin)


         Furniture  Office Supplies  Technology       TOTAL
Central  163797.16        167026.42   170416.31   501239.89
East     208291.20        205516.06   264973.98   678781.24
South    117298.68        125651.31   148771.91   391721.90
West     252612.74        220853.25   251991.83   725457.82
TOTAL    741999.80        719047.03   836154.03  2297200.86


In [31]:
# fill_value ( nan vlaue handle)
pt = df.pivot_table(values='Sales',
                    index='Region',
                    columns='Ship Mode',
                    aggfunc='sum',
                    fill_value=0).round(2)
print(pt)

Ship Mode  First Class  Same Day  Second Class  Standard Class
Region                                                        
Central       58746.92  20415.41     103550.01       318527.56
East         113587.05  43326.83     116545.52       405321.83
South         49332.57  21017.17      93758.61       227613.55
West         129761.89  43603.71     145339.43       406752.80


In [35]:
# multi index + mulitple columns
# region + segment rows  + category columns
pt_multi_idx = df.pivot_table(
    values='Sales',
    index=['Region', 'Segment'],
    columns='Category',
    aggfunc='sum',
    fill_value=0
).round(2)
pt_multi_idx.columns.name = None
pt_multi_idx.index.name = None
print(pt_multi_idx)


                     Furniture  Office Supplies  Technology
Region  Segment                                            
Central Consumer      86229.22         93111.48    72690.74
        Corporate     52085.60         41137.70    64772.51
        Home Office   25482.34         32777.24    32953.07
East    Consumer     114211.80        101255.14   135441.23
        Corporate     64209.05         66474.74    69725.57
        Home Office   29870.36         37786.18    59807.19
South   Consumer      70800.20         59504.58    65276.19
        Corporate     29645.03         45930.17    46310.73
        Home Office   16853.45         20216.56    37184.99
West    Consumer     119808.09        110080.94   132991.75
        Corporate     83080.11         77133.86    65641.31
        Home Office   49724.55         33638.45    53358.77


In [ ]:
# Topic - 8.4 melt - wide to  long
# columns to row convert

# syntax
df.melt(id_vars=['col1','col2'],     # yeh columns fixed rahenge
        value_vars=['col3','col4'],   # yeh columns rows mein jaayenge
        var_name='new_col_name',      # column names ka naam
        value_name='new_val_name')    # values ka naam

In [37]:
wide = pd.DataFrame({
    'Student':  ['Ali', 'Raj', 'Priya'],
    'Math':     [85, 92, 78],
    'Science':  [90, 88, 95],
    'English':  [78, 85, 92]
})
print(wide)

  Student  Math  Science  English
0     Ali    85       90       78
1     Raj    92       88       85
2   Priya    78       95       92


In [39]:
# melt long format convert
long = wide.melt(id_vars = 'Student', value_vars = ['Math','Science','English'],
                 var_name = 'Subject',
               value_name = 'Marks'  )
print(long)

  Student  Subject  Marks
0     Ali     Math     85
1     Raj     Math     92
2   Priya     Math     78
3     Ali  Science     90
4     Raj  Science     88
5   Priya  Science     95
6     Ali  English     78
7     Raj  English     85
8   Priya  English     92


In [40]:
# melt from dataset
pt = df.pivot_table(values='Sales',
                    index='Region',
                    columns='Category',
                    aggfunc='sum').round(2).reset_index()

print("Wide Format (Pivot Table):")
print(pt)

Wide Format (Pivot Table):
Category   Region  Furniture  Office Supplies  Technology
0         Central  163797.16        167026.42   170416.31
1            East  208291.20        205516.06   264973.98
2           South  117298.68        125651.31   148771.91
3            West  252612.74        220853.25   251991.83


In [43]:
# melt
long_format = pt.melt(id_vars='Region',
                      value_vars=['Furniture',
                                  'Office Supplies',
                                  'Technology'],
                      var_name='Category',
                      value_name='Total_Sales')

print(long_format)

     Region         Category  Total_Sales
0   Central        Furniture    163797.16
1      East        Furniture    208291.20
2     South        Furniture    117298.68
3      West        Furniture    252612.74
4   Central  Office Supplies    167026.42
5      East  Office Supplies    205516.06
6     South  Office Supplies    125651.31
7      West  Office Supplies    220853.25
8   Central       Technology    170416.31
9      East       Technology    264973.98
10    South       Technology    148771.91
11     West       Technology    251991.83


In [44]:
monthly = pd.DataFrame({
    'Product':  ['Laptop', 'Phone', 'Tablet'],
    'Jan_Sales': [50000, 30000, 20000],
    'Feb_Sales': [55000, 32000, 22000],
    'Mar_Sales': [60000, 35000, 25000],
    'Jan_Profit': [10000, 8000, 5000],
    'Feb_Profit': [11000, 8500, 5500],
    'Mar_Profit': [12000, 9000, 6000]
})
print(monthly)

  Product  Jan_Sales  Feb_Sales  Mar_Sales  Jan_Profit  Feb_Profit  Mar_Profit
0  Laptop      50000      55000      60000       10000       11000       12000
1   Phone      30000      32000      35000        8000        8500        9000
2  Tablet      20000      22000      25000        5000        5500        6000


In [46]:
# Sales columns melt
sales_long = monthly.melt(
    id_vars='Product',
    value_vars=['Jan_Sales', 'Feb_Sales', 'Mar_Sales'],
    var_name='Month',
    value_name='Sales'
)

# Month column clean karo
sales_long['Month'] = sales_long['Month'].str.replace('_Sales', '')

print("\nLong Format (Sales only):")
print(sales_long.sort_values('Product'))


Long Format (Sales only):
  Product Month  Sales
0  Laptop   Jan  50000
3  Laptop   Feb  55000
6  Laptop   Mar  60000
1   Phone   Jan  30000
4   Phone   Feb  32000
7   Phone   Mar  35000
2  Tablet   Jan  20000
5  Tablet   Feb  22000
8  Tablet   Mar  25000


In [48]:
# 8.5 — pivot_table() + melt() Together — Real Pipeline
import pandas as pd


# ── Step 1: Pivot Table banao — Analysis ke liye ──
print("=== STEP 1: PIVOT TABLE ===")
pt = df.pivot_table(
    values=['Sales', 'Profit'],
    index='Region',
    columns='Category',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='TOTAL'
).round(2)
print(pt)

# ── Step 2: Best performing Region+Category ──
print("\n=== STEP 2: BEST COMBINATIONS ===")
pt_sales = df.pivot_table(
    values='Sales',
    index='Region',
    columns='Category',
    aggfunc='sum'
).round(2)

# Stack karo — sabse zyada sales wali combination
# 🔑 Hard Word: stack()
# stack() = columns ko rows mein convert karo
# unstack() ka ulta — MultiIndex Series banta hai
stacked = pt_sales.stack().reset_index()
stacked.columns = ['Region', 'Category', 'Sales']
best = stacked.sort_values('Sales', ascending=False)
print("Top 5 Region-Category combinations:")
print(best.head(5))

# ── Step 3: Profit Margin Pivot ──
print("\n=== STEP 3: PROFIT MARGIN % ===")
sales_pt = df.pivot_table(values='Sales',
                          index='Region',
                          columns='Segment',
                          aggfunc='sum')
profit_pt = df.pivot_table(values='Profit',
                           index='Region',
                           columns='Segment',
                           aggfunc='sum')

margin_pt = (profit_pt / sales_pt * 100).round(2)
print("Profit Margin % by Region & Segment:")
print(margin_pt)

# ── Step 4: melt() se Long format — Visualization ready ──
print("\n=== STEP 4: MELT FOR VISUALIZATION ===")
pt_reset = pt_sales.reset_index()
long = pt_reset.melt(
    id_vars='Region',
    value_vars=['Furniture','Office Supplies','Technology'],
    var_name='Category',
    value_name='Sales'
)
print("Long format (ready for plotting):")
print(long)

# ── Step 5: Year-wise trend ──
print("\n=== STEP 5: YEAR-WISE TREND ===")
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Year'] = df['Order Date'].dt.year

year_cat = df.pivot_table(
    values='Sales',
    index='Year',
    columns='Category',
    aggfunc='sum',
    fill_value=0
).round(2)
print(year_cat)

=== STEP 1: PIVOT TABLE ===
            Profit                                            Sales  \
Category Furniture Office Supplies Technology      TOTAL  Furniture   
Region                                                                
Central   -2871.05         8879.98   33697.43   39706.36  163797.16   
East       3046.17        41014.58   47462.04   91522.78  208291.20   
South      6771.21        19986.39   19991.83   46749.43  117298.68   
West      11504.95        52609.85   44303.65  108418.45  252612.74   
TOTAL     18451.27       122490.80  145454.95  286397.02  741999.80   

                                                 
Category Office Supplies Technology       TOTAL  
Region                                           
Central        167026.42  170416.31   501239.89  
East           205516.06  264973.98   678781.24  
South          125651.31  148771.91   391721.90  
West           220853.25  251991.83   725457.82  
TOTAL          719047.03  836154.03  2297200.86  

==